# 面试问题：LLM Serving 怎样处理取消、背压，并保证 KV Block 一定释放？

        ## 可直接复述的回答主线

        1. 请求断开后如果仍生成到 max_tokens，会继续占用算力和 KV Cache，并让后续请求发生 OOM。
2. Serving 应把取消作为一等事件，同时从 active set 或等待队列移除请求，并通过统一 release 路径归还 KV block。
3. 资源不足时先进入有界队列形成背压；队列已满才明确拒绝，不能无界堆积或直接冲击 allocator。
4. 调度器需要维护 request_id、arrival、KV block、start/end、cancel 和 release 事件账本，方便证明没有双重释放或泄漏。
5. 同一批请求要比较完成/取消正确率、OOM 拒绝、队列峰值和取消后的浪费 block-tick。
6. 生产还需增量 KV 分配、客户端断连传播、CUDA 事件同步、抢占、公平性、限流与泄漏告警。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例模拟八条聊天生成请求，KV 池只有 8 个 block、等待队列容量为 3。两条请求会在生成中或排队时取消；其余请求具有不同 prompt、max_new_tokens 和服务时长。固定 tick 事件用于教学，不代表真实 GPU 时间。

In [1]:
import math  # 根据 prompt 和最大生成长度估算教学 KV Block 数。
block_size = 16  # 设定每个教学 KV Block 容纳十六个 Token。
kv_capacity = 8  # 设定整个副本只能同时持有八个 Block。
queue_capacity = 3  # 设定有界等待队列最多容纳三条请求。
request_specs = [{"id": "serve-01", "arrival": 0, "prompt_tokens": 32, "max_new_tokens": 16, "service_ticks": 6, "cancel_at": 2, "expected": "canceled"}, {"id": "serve-02", "arrival": 0, "prompt_tokens": 32, "max_new_tokens": 16, "service_ticks": 4, "cancel_at": None, "expected": "completed"}, {"id": "serve-03", "arrival": 1, "prompt_tokens": 48, "max_new_tokens": 16, "service_ticks": 4, "cancel_at": None, "expected": "completed"}, {"id": "serve-04", "arrival": 1, "prompt_tokens": 32, "max_new_tokens": 16, "service_ticks": 3, "cancel_at": None, "expected": "completed"}, {"id": "serve-05", "arrival": 2, "prompt_tokens": 48, "max_new_tokens": 16, "service_ticks": 4, "cancel_at": None, "expected": "completed"}, {"id": "serve-06", "arrival": 3, "prompt_tokens": 16, "max_new_tokens": 16, "service_ticks": 2, "cancel_at": None, "expected": "completed"}, {"id": "serve-07", "arrival": 4, "prompt_tokens": 32, "max_new_tokens": 16, "service_ticks": 4, "cancel_at": 5, "expected": "canceled"}, {"id": "serve-08", "arrival": 5, "prompt_tokens": 16, "max_new_tokens": 16, "service_ticks": 2, "cancel_at": None, "expected": "completed"}]  # 定义八条具有到达、资源、服务时长和取消语义的请求。
requests = [{**spec, "blocks": math.ceil((spec["prompt_tokens"] + spec["max_new_tokens"]) / block_size)} for spec in request_specs]  # 计算每条请求预留的教学 KV Block 数。
request_by_id = {request["id"]: request for request in requests}  # 建立请求 ID 到完整字段的索引。
print("教学实验输入：八条 LLM Serving 请求")  # 标记下方为确定性调度事件。
print("请求       arrival  prompt  max_new  blocks  service  cancel_at  expected")  # 输出请求预览表头。
for request in requests:  # 逐条展示到达和资源需求。
    print(f"{request['id']:<10} {request['arrival']:>7} {request['prompt_tokens']:>7} {request['max_new_tokens']:>8} {request['blocks']:>7} {request['service_ticks']:>8} {str(request['cancel_at']):>10}  {request['expected']}")  # 输出当前请求的调度字段。

教学实验输入：八条 LLM Serving 请求
请求       arrival  prompt  max_new  blocks  service  cancel_at  expected
serve-01         0      32       16       3        6          2  canceled
serve-02         0      32       16       3        4       None  completed
serve-03         1      48       16       4        4       None  completed
serve-04         1      32       16       3        3       None  completed
serve-05         2      48       16       4        4       None  completed
serve-06         3      16       16       2        2       None  completed
serve-07         4      32       16       3        4          5  canceled
serve-08         5      16       16       2        2       None  completed


## 2. Baseline / 基线：无队列，取消后继续生成

基线在到达时直接尝试预留全部 KV；不足就 OOM 拒绝。客户端取消只写日志，active 请求继续持有 Block 直到原计划结束。

In [2]:
def simulate_baseline(requests, capacity, horizon=15):  # 模拟忽略取消且没有背压队列的朴素 Serving。
    states = {request["id"]: {"status": "pending", "start": None, "end": None, "released": False, "client_canceled": False} for request in requests}  # 初始化每条请求状态。
    active = {}  # 保存已占用 KV 的活动请求。
    used_blocks = 0  # 记录当前 KV Block 占用。
    peak_blocks = 0  # 记录峰值 KV 占用。
    wasted_block_ticks = 0  # 统计客户端取消后仍占用的 Block 时间。
    ledger = []  # 保存 admit、OOM、cancel_ignored、complete 和 release 事件。
    for tick in range(horizon):  # 按固定逻辑时钟处理事件。
        completed_ids = [request_id for request_id, entry in active.items() if entry["end"] <= tick]  # 找到本 tick 正常结束的活动请求。
        for request_id in completed_ids:  # 逐条释放完成请求。
            entry = active.pop(request_id)  # 从活动集合移除请求。
            used_blocks -= entry["blocks"]  # 归还请求持有的 KV Block。
            final_status = "completed_after_cancel" if states[request_id]["client_canceled"] else "completed"  # 标记取消后仍生成完的浪费结果。
            states[request_id].update({"status": final_status, "end": tick, "released": True})  # 保存最终状态和释放时刻。
            ledger.append({"tick": tick, "event": "release", "id": request_id, "blocks": entry["blocks"], "used": used_blocks, "status": final_status})  # 记录统一完成释放事件。
        for request in requests:  # 检查本 tick 到达的取消信号。
            if request["cancel_at"] == tick and request["id"] in active:  # 识别活动请求的客户端断连。
                states[request["id"]]["client_canceled"] = True  # 记录客户端已取消但不释放资源。
                ledger.append({"tick": tick, "event": "cancel_ignored", "id": request["id"], "blocks": request["blocks"], "used": used_blocks})  # 展示错误的延迟取消行为。
        for request in [item for item in requests if item["arrival"] == tick]:  # 逐条处理本 tick 新到达请求。
            if used_blocks + request["blocks"] <= capacity:  # 检查能否立即预留全部 KV。
                used_blocks += request["blocks"]  # 分配请求所需 Block。
                end_tick = tick + request["service_ticks"]  # 计算忽略取消的计划结束时刻。
                active[request["id"]] = {"end": end_tick, "blocks": request["blocks"]}  # 把请求加入活动集合。
                states[request["id"]].update({"status": "active", "start": tick, "end": end_tick})  # 保存启动和计划结束信息。
                ledger.append({"tick": tick, "event": "admit", "id": request["id"], "blocks": request["blocks"], "used": used_blocks})  # 记录直接准入事件。
            else:  # 资源不足且没有等待队列。
                states[request["id"]]["status"] = "rejected_oom"  # 直接以 OOM 拒绝请求。
                ledger.append({"tick": tick, "event": "reject_oom", "id": request["id"], "blocks": request["blocks"], "used": used_blocks})  # 记录 OOM 拒绝。
        wasted_block_ticks += sum(entry["blocks"] for request_id, entry in active.items() if states[request_id]["client_canceled"])  # 累加已取消活动请求仍占用的 Block。
        peak_blocks = max(peak_blocks, used_blocks)  # 更新峰值 KV 占用。
    return states, ledger, {"peak_blocks": peak_blocks, "wasted_block_ticks": wasted_block_ticks, "oom_rejections": sum(state["status"] == "rejected_oom" for state in states.values())}  # 返回状态、账本和资源指标。
baseline_states, baseline_ledger, baseline_metrics = simulate_baseline(requests, kv_capacity)  # 在同一八条请求上运行朴素调度器。
print("Baseline 事件账本")  # 标记下表展示 OOM 和取消后继续持有。
for event in baseline_ledger:  # 逐事件展示资源变化。
    print(event)  # 输出当前逻辑时刻、请求、事件和 used blocks。
print("Baseline资源指标：", baseline_metrics)  # 展示峰值、OOM 和取消浪费。

Baseline 事件账本
{'tick': 0, 'event': 'admit', 'id': 'serve-01', 'blocks': 3, 'used': 3}
{'tick': 0, 'event': 'admit', 'id': 'serve-02', 'blocks': 3, 'used': 6}
{'tick': 1, 'event': 'reject_oom', 'id': 'serve-03', 'blocks': 4, 'used': 6}
{'tick': 1, 'event': 'reject_oom', 'id': 'serve-04', 'blocks': 3, 'used': 6}
{'tick': 2, 'event': 'cancel_ignored', 'id': 'serve-01', 'blocks': 3, 'used': 6}
{'tick': 2, 'event': 'reject_oom', 'id': 'serve-05', 'blocks': 4, 'used': 6}
{'tick': 3, 'event': 'admit', 'id': 'serve-06', 'blocks': 2, 'used': 8}
{'tick': 4, 'event': 'release', 'id': 'serve-02', 'blocks': 3, 'used': 5, 'status': 'completed'}
{'tick': 4, 'event': 'admit', 'id': 'serve-07', 'blocks': 3, 'used': 8}
{'tick': 5, 'event': 'release', 'id': 'serve-06', 'blocks': 2, 'used': 6, 'status': 'completed'}
{'tick': 5, 'event': 'cancel_ignored', 'id': 'serve-07', 'blocks': 3, 'used': 6}
{'tick': 5, 'event': 'admit', 'id': 'serve-08', 'blocks': 2, 'used': 8}
{'tick': 6, 'event': 'release', 'id': '

## 3. 底层实现：有界队列、active/queued 取消与统一 release

修正调度器先处理完成和取消，再从 FIFO 队列准入；新请求在资源不足时进入容量为 3 的队列。`release_active` 是唯一减少 KV 计数的路径，并阻止双重释放。

In [3]:
def simulate_corrected(requests, capacity, max_queue, horizon=15):  # 模拟具备取消传播、背压和 KV 回收的调度器。
    states = {request["id"]: {"status": "pending", "start": None, "end": None, "released": False} for request in requests}  # 初始化每条请求状态。
    active = {}  # 保存当前持有 KV 的请求。
    waiting = []  # 保存有界 FIFO 等待队列中的请求 ID。
    used_blocks = 0  # 记录当前 KV Block 占用。
    peak_blocks = 0  # 记录峰值 KV Block 占用。
    peak_queue = 0  # 记录队列峰值长度。
    ledger = []  # 保存排队、准入、取消、完成和释放事件。
    def release_active(request_id, tick, final_status):  # 通过唯一出口释放一个活动请求的 KV。
        nonlocal used_blocks  # 允许内部函数更新外层 Block 计数。
        if request_id not in active or states[request_id]["released"]:  # 防止不存在请求或双重释放。
            return False  # 返回未执行释放。
        entry = active.pop(request_id)  # 从 active 集合原子移除请求。
        used_blocks -= entry["blocks"]  # 精确归还该请求持有的 KV Block。
        states[request_id].update({"status": final_status, "end": tick, "released": True})  # 保存最终状态和释放时刻。
        ledger.append({"tick": tick, "event": "release", "id": request_id, "blocks": entry["blocks"], "used": used_blocks, "status": final_status})  # 记录可审计的单次释放。
        return True  # 返回释放已执行。
    def admit(request_id, tick):  # 为一条请求分配 KV 并开始服务。
        nonlocal used_blocks  # 允许内部函数更新 Block 计数。
        request = request_by_id[request_id]  # 读取完整请求字段。
        used_blocks += request["blocks"]  # 分配请求所需 KV Block。
        end_tick = tick + request["service_ticks"]  # 计算从实际准入时刻开始的结束时间。
        active[request_id] = {"blocks": request["blocks"], "end": end_tick}  # 把请求加入活动集合。
        states[request_id].update({"status": "active", "start": tick, "end": end_tick})  # 保存启动状态和计划结束。
        ledger.append({"tick": tick, "event": "admit", "id": request_id, "blocks": request["blocks"], "used": used_blocks, "queue": list(waiting)})  # 记录准入及剩余队列。
    def drain_queue(tick):  # 在有可用 Block 时按 FIFO 尝试准入等待请求。
        while waiting:  # 只要队列非空就检查队首。
            request_id = waiting[0]  # 读取 FIFO 队首请求。
            request = request_by_id[request_id]  # 读取队首 Block 需求。
            if used_blocks + request["blocks"] > capacity:  # 检查队首是否仍无法容纳。
                break  # 保持公平顺序并等待下次释放。
            waiting.pop(0)  # 从等待队列移除即将准入的请求。
            admit(request_id, tick)  # 分配 KV 并开始执行。
    for tick in range(horizon):  # 按固定逻辑时钟处理完整事件流。
        completed_ids = [request_id for request_id, entry in active.items() if entry["end"] <= tick]  # 找到本 tick 完成的活动请求。
        for request_id in completed_ids:  # 逐完成请求走统一 release。
            release_active(request_id, tick, "completed")  # 正常完成并释放 KV。
        for request in requests:  # 检查活动和排队请求的取消信号。
            if request["cancel_at"] != tick:  # 跳过本 tick 没有取消的请求。
                continue  # 继续检查下一条请求。
            if request["id"] in active:  # 处理生成中的客户端断连。
                release_active(request["id"], tick, "canceled")  # 立即停止并通过统一路径释放 KV。
            elif request["id"] in waiting:  # 处理尚未准入的排队取消。
                waiting.remove(request["id"])  # 从有界队列移除已无消费者的请求。
                states[request["id"]].update({"status": "canceled", "end": tick, "released": True})  # 标记排队取消无需 KV 释放。
                ledger.append({"tick": tick, "event": "cancel_queued", "id": request["id"], "blocks": 0, "used": used_blocks, "queue": list(waiting)})  # 记录队列槽位被归还。
        drain_queue(tick)  # 在完成和取消释放后优先服务旧队列。
        for request in [item for item in requests if item["arrival"] == tick]:  # 逐条处理本 tick 新到达请求。
            if not waiting and used_blocks + request["blocks"] <= capacity:  # 无旧队列且资源足够时直接准入。
                admit(request["id"], tick)  # 分配 KV 并开始执行。
            elif len(waiting) < max_queue:  # 资源不足时检查有界队列容量。
                waiting.append(request["id"])  # 把请求加入 FIFO 队尾形成背压。
                states[request["id"]]["status"] = "queued"  # 保存排队状态。
                ledger.append({"tick": tick, "event": "enqueue", "id": request["id"], "blocks": request["blocks"], "used": used_blocks, "queue": list(waiting)})  # 记录队列长度和资源快照。
            else:  # KV 和等待队列都没有容量。
                states[request["id"]]["status"] = "rejected_backpressure"  # 明确返回背压拒绝而不是触发 OOM。
                ledger.append({"tick": tick, "event": "reject_backpressure", "id": request["id"], "blocks": request["blocks"], "used": used_blocks, "queue": list(waiting)})  # 记录有界拒绝。
        drain_queue(tick)  # 让本 tick 新入队且可容纳的队首立即开始。
        peak_blocks = max(peak_blocks, used_blocks)  # 更新峰值 KV 占用。
        peak_queue = max(peak_queue, len(waiting))  # 更新峰值等待队列长度。
    metrics = {"peak_blocks": peak_blocks, "peak_queue": peak_queue, "backpressure_rejections": sum(state["status"] == "rejected_backpressure" for state in states.values()), "canceled": sum(state["status"] == "canceled" for state in states.values()), "released_active": sum(event["event"] == "release" for event in ledger), "wasted_block_ticks_after_cancel": 0, "final_used_blocks": used_blocks, "final_active": len(active), "final_waiting": len(waiting)}  # 汇总资源、背压、取消和最终回收指标。
    return states, ledger, metrics  # 返回最终状态、事件账本和服务指标。
corrected_states, corrected_ledger, corrected_metrics = simulate_corrected(requests, kv_capacity, queue_capacity)  # 在同一事件流上运行修正调度器。
print("修正调度关键事件账本")  # 标记下表展示排队、取消和统一释放。
for event in corrected_ledger:  # 逐事件展示状态迁移和 Block 计数。
    print(event)  # 输出当前调度事件的完整可读字段。

修正调度关键事件账本
{'tick': 0, 'event': 'admit', 'id': 'serve-01', 'blocks': 3, 'used': 3, 'queue': []}
{'tick': 0, 'event': 'admit', 'id': 'serve-02', 'blocks': 3, 'used': 6, 'queue': []}
{'tick': 1, 'event': 'enqueue', 'id': 'serve-03', 'blocks': 4, 'used': 6, 'queue': ['serve-03']}
{'tick': 1, 'event': 'enqueue', 'id': 'serve-04', 'blocks': 3, 'used': 6, 'queue': ['serve-03', 'serve-04']}
{'tick': 2, 'event': 'release', 'id': 'serve-01', 'blocks': 3, 'used': 3, 'status': 'canceled'}
{'tick': 2, 'event': 'admit', 'id': 'serve-03', 'blocks': 4, 'used': 7, 'queue': ['serve-04']}
{'tick': 2, 'event': 'enqueue', 'id': 'serve-05', 'blocks': 4, 'used': 7, 'queue': ['serve-04', 'serve-05']}
{'tick': 3, 'event': 'enqueue', 'id': 'serve-06', 'blocks': 2, 'used': 7, 'queue': ['serve-04', 'serve-05', 'serve-06']}
{'tick': 4, 'event': 'release', 'id': 'serve-02', 'blocks': 3, 'used': 4, 'status': 'completed'}
{'tick': 4, 'event': 'admit', 'id': 'serve-04', 'blocks': 3, 'used': 7, 'queue': ['serve-05', '

## 4. 逐请求结果与结果解读

正确结果要求取消请求为 `canceled`，其余请求最终 `completed`。修正方案利用释放后的 Block 逐步排空队列，不靠超配 KV。

In [4]:
baseline_rows = []  # 保存八条请求的基线最终结果。
corrected_rows = []  # 保存八条请求的修正最终结果。
for request in requests:  # 逐请求对照状态和期望。
    baseline_status = baseline_states[request["id"]]["status"]  # 读取基线最终状态。
    corrected_status = corrected_states[request["id"]]["status"]  # 读取修正最终状态。
    baseline_rows.append({"id": request["id"], "status": baseline_status, "correct": baseline_status == request["expected"]})  # 保存基线正确性。
    corrected_rows.append({"id": request["id"], "status": corrected_status, "correct": corrected_status == request["expected"], "start": corrected_states[request["id"]]["start"], "end": corrected_states[request["id"]]["end"]})  # 保存修正状态和服务时刻。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(baseline_rows)  # 计算基线生命周期正确率。
corrected_accuracy = sum(row["correct"] for row in corrected_rows) / len(corrected_rows)  # 计算修正生命周期正确率。
final_used_blocks = corrected_metrics["final_used_blocks"]  # 读取调度器真实结束时的 KV Block 计数。
print("请求       expected    baseline                 corrected   start/end  correct")  # 输出逐请求同数据对照表头。
for request, baseline, corrected in zip(requests, baseline_rows, corrected_rows):  # 逐请求展示取消、OOM、排队和完成结果。
    print(f"{request['id']:<10} {request['expected']:<11} {baseline['status']:<24} {corrected['status']:<11} {str((corrected['start'], corrected['end'])):<11} {str(corrected['correct']):>7}")  # 输出当前请求完整生命周期。
print(f"结果解读：生命周期正确率从{baseline_accuracy:.1%}升到{corrected_accuracy:.1%}；Baseline有{baseline_metrics['oom_rejections']}次OOM拒绝和{baseline_metrics['wasted_block_ticks']}个取消后浪费block-tick，修正后均为0。")  # 解释取消回收和背压带来的服务收益。

请求       expected    baseline                 corrected   start/end  correct
serve-01   canceled    completed_after_cancel   canceled    (0, 2)         True
serve-02   completed   completed                completed   (0, 4)         True
serve-03   completed   rejected_oom             completed   (2, 6)         True
serve-04   completed   rejected_oom             completed   (4, 7)         True
serve-05   completed   rejected_oom             completed   (6, 10)        True
serve-06   completed   completed                completed   (7, 9)         True
serve-07   canceled    completed_after_cancel   canceled    (None, 5)      True
serve-08   completed   completed                completed   (7, 9)         True
结果解读：生命周期正确率从37.5%升到100.0%；Baseline有3次OOM拒绝和21个取消后浪费block-tick，修正后均为0。


## 5. 失败案例与修正：只取消 active，不清理 queued

`serve-07` 在 tick=5 仍处于队列。若取消处理只扫描 active，它会继续占满第三个队列槽，使同 tick 到达的 `serve-08` 被错误拒绝；移除 queued 条目后可以正常入队。

In [5]:
stale_queue = ["serve-05", "serve-06", "serve-07"]  # 构造 tick=5 取消前的满队列快照。
naive_queue_after_cancel = stale_queue.copy()  # 错误实现没有从 waiting 中删除 serve-07。
naive_accepts_serve08 = len(naive_queue_after_cancel) < queue_capacity  # 满队列会错误拒绝新请求。
fixed_queue_after_cancel = [request_id for request_id in stale_queue if request_id != "serve-07"]  # 修正实现移除排队取消请求。
fixed_accepts_serve08 = len(fixed_queue_after_cancel) < queue_capacity  # 释放槽位后新请求可以进入背压队列。
if fixed_accepts_serve08:  # 检查修正队列确实有容量。
    fixed_queue_after_cancel.append("serve-08")  # 模拟同 tick 新请求安全入队。
print(f"错误行为：queued取消未清理，queue={naive_queue_after_cancel}，serve-08 accepted={naive_accepts_serve08}")  # 展示陈旧请求占用队列容量。
print(f"修正行为：移除serve-07后queue={fixed_queue_after_cancel}，serve-08 accepted={fixed_accepts_serve08}")  # 展示等待队列取消传播。

错误行为：queued取消未清理，queue=['serve-05', 'serve-06', 'serve-07']，serve-08 accepted=False
修正行为：移除serve-07后queue=['serve-05', 'serve-06', 'serve-08']，serve-08 accepted=True


## 6. 生产边界

本例为简化而预留最大 KV，真实引擎会随 decode 增量分配。生产需要连接断开到 scheduler 的低延迟传播、CUDA kernel 安全点、连续批处理移除、block 引用计数、幂等 release、租户限流、队列 deadline、抢占与 p99 排队/泄漏监控。

In [6]:
serving_diagnostics = {"requests": len(requests), "kv_capacity": kv_capacity, "queue_capacity": queue_capacity, "baseline_accuracy": baseline_accuracy, "corrected_accuracy": corrected_accuracy, "baseline_oom_rejections": baseline_metrics["oom_rejections"], "baseline_wasted_block_ticks": baseline_metrics["wasted_block_ticks"], "corrected_peak_queue": corrected_metrics["peak_queue"], "corrected_backpressure_rejections": corrected_metrics["backpressure_rejections"], "final_used_blocks": final_used_blocks}  # 汇总容量、取消、背压和泄漏指标。
print("生产监控快照：", serving_diagnostics)  # 输出 Serving 调度器应持续观察的资源信号。

生产监控快照： {'requests': 8, 'kv_capacity': 8, 'queue_capacity': 3, 'baseline_accuracy': 0.375, 'corrected_accuracy': 1.0, 'baseline_oom_rejections': 3, 'baseline_wasted_block_ticks': 21, 'corrected_peak_queue': 3, 'corrected_backpressure_rejections': 0, 'final_used_blocks': 0}


## 7. 最小回归测试

断言覆盖样本规模、容量上限、取消、生命周期结果、队列清理和最终 KV 释放。

In [7]:
assert len(requests) >= 5 and kv_capacity == 8  # 保证案例包含足够请求且容量约束明确。
assert baseline_metrics["peak_blocks"] <= kv_capacity and corrected_metrics["peak_blocks"] <= kv_capacity  # 保证两种调度都没有越过物理 KV 容量。
assert baseline_metrics["wasted_block_ticks"] > 0 and corrected_metrics["wasted_block_ticks_after_cancel"] == 0  # 保证取消后继续生成的浪费真实复现并消除。
assert corrected_accuracy > baseline_accuracy and corrected_accuracy == 1.0  # 保证同一事件流的生命周期结果全面改善。
assert all(corrected_states[request_id]["released"] for request_id in ("serve-01", "serve-02", "serve-03", "serve-04", "serve-05", "serve-06", "serve-07", "serve-08"))  # 保证完成和取消请求都经过资源终态。
assert not naive_accepts_serve08 and fixed_accepts_serve08 and "serve-07" not in fixed_queue_after_cancel  # 保证 queued 取消槽位泄漏被修正。
assert final_used_blocks == 0 and corrected_metrics["backpressure_rejections"] == 0  # 保证仿真结束没有 KV 泄漏且有界队列吸收突发。